# Graph Matching

This notebook runs the phase-one pipeline: snapshot construction, Louvain community detection, and Jaccard-based community matching. Persistent IDs and lifecycle event classification are intentionally left for a later phase.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "dataset" / "email-Eu-core-temporal.txt").exists():
            return candidate
    raise FileNotFoundError("Could not find dataset/email-Eu-core-temporal.txt in this folder or its parents.")

PROJECT_DIR = find_project_root()
CODE_DIR = Path.cwd().resolve()
if not (CODE_DIR / "graph_matching.py").exists():
    CODE_DIR = PROJECT_DIR / "code" / "graph-matching"

sys.path.insert(0, str(CODE_DIR))

PROJECT_DIR, CODE_DIR

(PosixPath('/Users/maha_personal/Uni/SS 26/IDP'),
 PosixPath('/Users/maha_personal/Uni/SS 26/IDP/code/graph-matching'))

In [2]:
import pandas as pd

from graph_matching import load_edges, run_approach

DATA_PATH = PROJECT_DIR / "dataset" / "email-Eu-core-temporal.txt"
OUTPUT_DIR = CODE_DIR / "outputs" / "graph_matching"

edges = load_edges(DATA_PATH, cutoff_days=500)
edges.head(), len(edges), edges["ts"].max() / (24 * 60 * 60)

(   src  dst    ts
 0  582  364     0
 1  168  472  2797
 2  168  912  3304
 3    2  790  4523
 4    2  322  7926,
 307868,
 np.float64(499.9850925925926))

In [3]:
for approach in ["cumulative", "interval", "overlap"]:
    run_approach(
        edges=edges,
        approach=approach,
        output_dir=OUTPUT_DIR,
        cutoff_days=500,
        snapshot_days=50,
        num_snapshots=10,
        overlap_fraction=0.5,
        seed=42,
        resolution=1.0,
        min_community_size=3,
        match_threshold=0.3,
    )

OUTPUT_DIR

PosixPath('/Users/maha_personal/Uni/SS 26/IDP/code/graph-matching/outputs/graph_matching')

In [4]:
summary = []
for approach in ["cumulative", "interval", "overlap"]:
    base = OUTPUT_DIR / approach
    stats = pd.read_csv(base / "snapshot_stats.csv")
    communities = pd.read_csv(base / "communities.csv")
    matches = pd.read_csv(base / "matches.csv")
    summary.append({
        "approach": approach,
        "snapshots": len(stats),
        "communities": len(communities),
        "matches": len(matches),
        "mean_jaccard": matches["jaccard"].mean(),
        "min_jaccard": matches["jaccard"].min(),
        "max_jaccard": matches["jaccard"].max(),
    })

pd.DataFrame(summary).round(4)


,approach,snapshots,communities,matches,mean_jaccard,min_jaccard,max_jaccard
0,cumulative,10,132,108,0.8013,0.3014,1.0000
1,interval,10,144,96,0.5399,0.3012,0.8704
2,overlap,19,271,228,0.6219,0.3000,1.0000
